In [ ]:
import requests
import time
from bs4 import BeautifulSoup
from urllib.parse import urljoin
import re

zhe_cookie = input("\nEnter ZHE cookie: ").strip()
phpsessid_cookie = input("Enter PHPSESSID cookie: ").strip()

cookie = {
    "ZHE": zhe_cookie,
    "PHPSESSID": phpsessid_cookie
}

headers = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36"
}

def grab_archive_sites():
    # Cập nhật base URL chính xác theo ảnh
    base = "https://zone-h.org"

    for i in range(1, 20):
        url = f"{base}/archive/special={i}"
        print(f"\n[+] Đang lấy danh sách từ: {url}")

        try:
            res = requests.get(url, cookies=cookie, headers=headers, timeout=10)
        except requests.exceptions.RequestException as e:
            print(f"Lỗi request: {e}")
            continue

        if res.status_code != 200:
            print(f"Status code lỗi: {res.status_code}")
            continue

        html = res.text

        if "captcha" in html.lower():
            print("\n!!! CAPTCHA detected tại trang danh sách !!!")
            input("Giải CAPTCHA trên trình duyệt rồi nhấn Enter...")
            
            new_zhe = input("Nhập lại ZHE cookie (nhấn Enter nếu giữ nguyên): ").strip()
            new_php = input("Nhập lại PHPSESSID cookie (nhấn Enter nếu giữ nguyên): ").strip()
            if new_zhe: cookie["ZHE"] = new_zhe
            if new_php: cookie["PHPSESSID"] = new_php
            continue

        soup = BeautifulSoup(html, "html.parser")

        # Bước 1: Lọc link CHÍNH XÁC cấu trúc "mirror/id/" (không có dấu "=")
        mirror_links = []
        for a in soup.find_all("a", href=True):
            href_value = a["href"]
            if "mirror/id/" in href_value:
                full_mirror_url = urljoin(base, href_value)
                if full_mirror_url not in mirror_links:
                    mirror_links.append(full_mirror_url)

        print(f"Tìm thấy {len(mirror_links)} link mirror. Bắt đầu lấy domain...")

        # Bước 2: Vào từng trang mirror để bóc tách full url
        with open("full_urls.txt", "a", encoding="utf-8") as f:
            for mirror_url in mirror_links:
                try:
                    m_res = requests.get(mirror_url, cookies=cookie, headers=headers, timeout=10)

                    if "captcha" in m_res.text.lower():
                        print(f"\n!!! CAPTCHA detected tại trang mirror {mirror_url} !!!")
                        print("Tạm dừng, vui lòng kiểm tra lại trình duyệt.")
                        continue

                    # Dùng BeautifulSoup gom text lại và dùng Regex quét chữ "Domain:"
                    m_soup = BeautifulSoup(m_res.text, "html.parser")
                    text_content = m_soup.get_text(separator=' ')

                    # Regex tìm chữ "Domain:" và lấy cụm ký tự liền kề sau đó
                    domain_match = re.search(r'Domain:\s*(https?://[^\s]+|[^\s]+)', text_content, re.IGNORECASE)

                    if domain_match:
                        full_domain = domain_match.group(1).strip()
                        print(f"  [V] Lấy thành công: {full_domain}")
                        f.write(full_domain + "\n")
                    else:
                        print(f"  [X] Không tìm thấy domain trong: {mirror_url}")

                except requests.exceptions.RequestException as e:
                    print(f"  [!] Lỗi khi truy cập mirror: {e}")

                # Chờ 3 giây giữa các lần quét mirror để tránh bị khóa IP/Cookie
                time.sleep(3)

        time.sleep(2)

def main():
    grab_archive_sites()

if __name__ == "__main__":
    main()